In [2]:
import nest_asyncio
nest_asyncio.apply()

import os
import glob
import pandas as pd
import seaborn as sns
import dataframe_image as dfi
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

In [ ]:
# --- Configuration ---

# The path to the folder the your CSV files.
FOLDER_PATH = '/fast/AG_Kainmueller/vguarin/aggrigator_experiments/output/tables/auroc_gmm/'

# This dictionary maps the aggregator names in the CSV files
AGGREGATOR_NAME_MAPPING = {
    'Mean': 'AVG',
    'Quantile 0.6': 'AQA 0.60',
    'Quantile 0.75': 'AQA 0.75',
    'Quantile 0.9': 'AQA 0.90',
    'Patch 10': 'PLM 10',
    'Patch 20': 'PLM 20',
    'Patch 50': 'PLM 50',
    'Threshold 0.3': 'ATA 0.3',
    'Threshold 0.5': 'ATA 0.5',
    'Threshold 0.7': 'ATA 0.7',
    'Quantile fg. ratio': 'QFR',
    'Imbalance-w. class avg.': 'ICA', # Assuming this mapping
    'Equally-w. class avg.': 'BCA', # Assuming this mapping
    'GMM_pixel': 'GMM-Int',           # Assuming this mapping
    'GMM_spatial': 'GMM-Spa',         # Assuming this mapping
    'GMM': 'GMM-All',                 # Assuming this mapping
}

COLUMN_NAME_MAPPING = {
    'instance_lizard_glas_set_pu': 'LIZ-IG',
    'fgbg_wormbodies_protists_pu': 'WORM-Pro',
    'instance_arctique_nuclei_intensity_pu': 'ARC-Nuc',
    'semantic_gta_cityscapes_pu': 'CAR-CS',
    'crops_vs_weed_weedsgalore_maize_pu': 'WEED-Hand',
    'fgbg_wormbodies_nematodes_pu': 'WORM-Nem',
    'fgbg_lidc_malignancy_pu': 'LIDC-Mal',
    'semantic_lizard_glas_set_pu': 'LIZ-SG',
    'fgbg_lidc_texture_pu': 'LIDC-Tex',
    'semantic_arctique_blood_cells_pu': 'ARC-BC'
}

# --- Data Loading and Processing ---

# Find all relevant CSV files in the specified folder
search_pattern = os.path.join(FOLDER_PATH, '*_auroc_ood_results.csv')
file_paths = glob.glob(search_pattern)

if not file_paths:
    print(f"Error: No CSV files found at '{search_pattern}'. Please check your FOLDER_PATH.")
else:
    print(f"Found {len(file_paths)} CSV files to process.")

all_means = []
all_stds = []

for filepath in file_paths:
    # Extract the dataset name from the filename.
    basename = os.path.basename(filepath)
    dataset_name = basename.replace('_auroc_ood_results.csv', '')

    # Read AUROC and AUROC_std ---
    temp_df = pd.read_csv(filepath)
    temp_df['Aggregator'] = temp_df['Aggregator'].map(AGGREGATOR_NAME_MAPPING).fillna(temp_df['Aggregator'])
    temp_df = temp_df.set_index('Aggregator')

    # Create and append the means DataFrame
    mean_df = temp_df[['AUROC']].rename(columns={'AUROC': dataset_name})
    all_means.append(mean_df)
    
    std_df = temp_df[['AUROC_std']].rename(columns={'AUROC_std': dataset_name})
    all_stds.append(std_df)
    
# Numeric DataFrame for calculations and color mapping
summary_df = pd.concat(all_means, axis=1)
stds_df = pd.concat(all_stds, axis=1)

summary_df = summary_df.rename(columns=COLUMN_NAME_MAPPING)
stds_df = stds_df.rename(columns=COLUMN_NAME_MAPPING)

# --- Calculate Rank, Reorder, and Sort ---
ranks_df = summary_df.rank(ascending=False, method='min')
summary_df['Avg Rank'] = ranks_df.mean(axis=1)
final_column_order = [
    'ARC-BC', 'ARC-Nuc', 'CAR-CS', 'LIDC-Mal', 'LIDC-Tex',
    'LIZ-IG', 'LIZ-SG', 'WEED-Hand', 'WORM-Nem', 'WORM-Pro', 'Avg Rank'
]
summary_df = summary_df[final_column_order]
summary_df = summary_df.sort_values(by='Avg Rank', ascending=True)

# Align stds_df to the final sorted summary_df
stds_df = stds_df.reindex(index=summary_df.index, columns=summary_df.columns.drop('Avg Rank', errors='ignore'))
summary_df.index.name = None

# Create a second DataFrame with the desired string formats for display
display_df = pd.DataFrame(index=summary_df.index, columns=summary_df.columns, dtype=str)
for col in display_df.columns:
    if col == 'Avg Rank':
        display_df[col] = summary_df[col].map('{:.1f}'.format)
    else:
        # Create "mean ± std" strings
        mean_series = summary_df[col]
        std_series = stds_df[col]
        display_df[col] = mean_series.map('{:.2f}'.format) + ' ± ' + std_series.map('{:.2f}'.format)

# --- Final Styling ---
dataset_cols = [col for col in summary_df.columns if col != 'Avg Rank']

# Apply heatmap styling similar to the example image
# Green for high values, red for low, white for middle
styled_df = display_df.style.background_gradient(
    cmap=sns.diverging_palette(10, 130, as_cmap=True), # Red to Green palette #'RdYlGn',
    gmap=summary_df[dataset_cols],
    axis=None, 
    low=0.3, # Adjust these to control the color intensity
    high=0.7
).set_properties(
    **{'width': '100px'}
)

print("\n--- Styled Summary Table with Mean Rank ---")
# Display the styled DataFrame in the notebook
display(styled_df)

# --- Save the DataFrame for Sharing ---

# You can save the data in several formats.

# a) Save the raw data (without styles) to a CSV file
output_csv_path = 'auroc_summary_with_ranks.csv'
summary_df.to_csv(output_csv_path)
print(f"\nSuccessfully saved data to '{output_csv_path}'")

# b) Export using the Matplotlib backend
output_image_path = 'auroc_summary_table.png'

dfi.export(
    styled_df,
    output_image_path,
    table_conversion='matplotlib' # Ensures we use the reliable backend
)

print(f"Successfully saved styled table as a PNG to '{output_image_path}' using the Matplotlib backend.")

Found 10 CSV files to process.

--- Styled Summary Table with Mean Rank ---


,ARC-BC,ARC-Nuc,CAR-CS,LIDC-Mal,LIDC-Tex,LIZ-IG,LIZ-SG,WEED-Hand,WORM-Nem,WORM-Pro,Avg Rank
BCA,0.90 ± 0.04,0.79 ± 0.05,0.89 ± 0.01,0.57 ± 0.05,0.82 ± 0.07,0.68 ± 0.02,0.68 ± 0.03,0.58 ± 0.07,0.77 ± 0.06,0.94 ± 0.02,5.3
ICA,0.82 ± 0.06,0.83 ± 0.05,0.84 ± 0.02,0.57 ± 0.05,0.82 ± 0.08,0.71 ± 0.02,0.59 ± 0.03,0.58 ± 0.06,0.77 ± 0.06,0.94 ± 0.02,5.5
GMM-All,0.84 ± 0.05,0.86 ± 0.05,1.00 ± 0.00,0.86 ± 0.03,0.77 ± 0.05,0.45 ± 0.03,0.44 ± 0.03,0.95 ± 0.03,1.00 ± 0.00,1.00 ± 0.00,5.6
QFR,0.89 ± 0.04,0.87 ± 0.05,0.62 ± 0.02,0.54 ± 0.05,0.88 ± 0.05,0.68 ± 0.02,0.57 ± 0.03,0.57 ± 0.07,0.68 ± 0.07,0.91 ± 0.03,6.2
GMM-Int,0.79 ± 0.06,0.83 ± 0.05,0.73 ± 0.02,0.86 ± 0.03,0.78 ± 0.05,0.49 ± 0.03,0.43 ± 0.03,0.91 ± 0.04,0.98 ± 0.01,1.00 ± 0.00,6.4
GMM-Spa,0.93 ± 0.03,0.66 ± 0.07,1.00 ± 0.00,0.67 ± 0.05,0.71 ± 0.07,0.49 ± 0.03,0.41 ± 0.03,0.85 ± 0.04,0.89 ± 0.04,0.90 ± 0.03,7.1
PLM 20,0.71 ± 0.07,0.68 ± 0.07,0.42 ± 0.03,0.95 ± 0.02,0.52 ± 0.08,0.67 ± 0.03,0.73 ± 0.02,0.57 ± 0.05,0.57 ± 0.07,0.84 ± 0.04,8.8
AQA 0.60,0.73 ± 0.06,0.65 ± 0.07,0.64 ± 0.02,0.95 ± 0.02,0.50 ± 0.08,0.76 ± 0.02,0.81 ± 0.02,0.33 ± 0.05,0.49 ± 0.07,0.55 ± 0.05,9.0
PLM 50,0.59 ± 0.08,0.69 ± 0.07,0.46 ± 0.02,0.95 ± 0.02,0.50 ± 0.08,0.68 ± 0.02,0.75 ± 0.02,0.57 ± 0.05,0.48 ± 0.07,0.86 ± 0.04,9.2
PLM 10,0.67 ± 0.07,0.65 ± 0.07,0.44 ± 0.03,0.91 ± 0.03,0.59 ± 0.09,0.65 ± 0.02,0.65 ± 0.02,0.50 ± 0.04,0.69 ± 0.06,0.85 ± 0.04,9.3



Successfully saved data to 'auroc_summary_with_ranks.csv'
Successfully saved styled table as a PNG to 'auroc_summary_table.png' using the Matplotlib backend.


In [9]:
# --- Configuration ---

# The path to the folder containing the EAURC CSV files.
FOLDER_PATH = '/fast/AG_Kainmueller/vguarin/aggrigator_experiments/output/tables/eaurc_id_ood/'

# This dictionary maps the aggregator names in the CSV files to the
# final names you want in the table.
AGGREGATOR_NAME_MAPPING = {
    'Mean': 'AVG',
    'Quantile 0.6': 'AQA 0.60',
    'Quantile 0.75': 'AQA 0.75',
    'Quantile 0.9': 'AQA 0.90',
    'Patch 10': 'PLM 10',
    'Patch 20': 'PLM 20',
    'Patch 50': 'PLM 50',
    'Threshold 0.3': 'ATA 0.3',
    'Threshold 0.5': 'ATA 0.5',
    'Threshold 0.7': 'ATA 0.7',
    'Quantile fg. ratio': 'QFR',
    'Imbalance-w. class avg.': 'ICA',
    'Equally-w. class avg.': 'BCA',
    'GMM_pixel': 'GMM-I',
    'GMM_spatial': 'GMM-S',
    'GMM': 'GMM-F',
}


# --- Data Loading and Processing ---

# Find all relevant CSV files in the specified folder
search_pattern = os.path.join(FOLDER_PATH, '*_eaurc_id_ood_results.csv')
file_paths = glob.glob(search_pattern)

if not file_paths:
    print(f"Error: No CSV files found at '{search_pattern}'. Please check your FOLDER_PATH.")
else:
    print(f"Found {len(file_paths)} CSV files to process.")

all_results = []

for filepath in file_paths:
    # Extract the dataset name from the filename
    basename = os.path.basename(filepath)
    dataset_name = basename.replace('_eaurc_id_ood_results.csv', '')

    # Read the CSV data
    temp_df = pd.read_csv(filepath)

    # Standardize aggregator names
    temp_df['Aggregator'] = temp_df['Aggregator'].map(AGGREGATOR_NAME_MAPPING).fillna(temp_df['Aggregator'])

    # Keep only the EAURC column
    temp_df = temp_df[['Aggregator', 'EAURC']].set_index('Aggregator')

    # Rename the column to the dataset name
    temp_df = temp_df.rename(columns={'EAURC': dataset_name})

    all_results.append(temp_df)

# Combine all results into a single DataFrame
summary_df_eaurc = pd.concat(all_results, axis=1)


# --- Calculate Mean Rank (where LOWEST is better) ---

# Rank each aggregator within each dataset column.
# *** CRITICAL CHANGE: ascending=True means lower EAURC gets a better rank (Rank 1) ***
ranks_df_eaurc = summary_df_eaurc.rank(ascending=True, method='min')

# Calculate the mean rank across all datasets for each aggregator
summary_df_eaurc['Mean Rank'] = ranks_df_eaurc.mean(axis=1)


# --- Final Formatting and Sorting ---

# Sort the table by the new 'Mean Rank' column (lower is better)
summary_df_eaurc = summary_df_eaurc.sort_values(by='Mean Rank', ascending=True)

# Separate data columns from the rank column for styling
dataset_cols_eaurc = [col for col in summary_df_eaurc.columns if col != 'Mean Rank']

# Apply heatmap styling
# *** CRITICAL CHANGE: The color map is reversed (Green for low values, Red for high) ***
styled_df_eaurc = summary_df_eaurc.style.background_gradient(
    cmap=sns.diverging_palette(130, 10, as_cmap=True), # Green to Red palette
    subset=dataset_cols_eaurc,
    low=0.3,
    high=0.7
).format(
    '{:.2f}', # Format EAURC values
    subset=dataset_cols_eaurc
).format(
    '{:.2f}', # Format Rank value
    subset=['Mean Rank']
).set_properties(**{'width': '100px'})

print("\n--- Styled EAURC Summary Table with Mean Rank (Lowest is Better) ---")
# Display the styled DataFrame in the notebook
display(styled_df_eaurc)


# --- Save the DataFrame for Sharing ---

# Save the raw data (without styles) to a CSV file
output_csv_path_eaurc = 'eaurc_summary_with_ranks.csv'
summary_df_eaurc.to_csv(output_csv_path_eaurc)
print(f"\nSuccessfully saved data to '{output_csv_path_eaurc}'")

# Optional: To save the styled table as an image
# import dataframe_image as dfi
# output_image_path_eaurc = 'eaurc_summary_table.png'
# dfi.export(styled_df_eaurc, output_image_path_eaurc)
# print(f"Successfully saved styled table image to '{output_image_path_eaurc}'")

Found 10 CSV files to process.

--- Styled EAURC Summary Table with Mean Rank (Lowest is Better) ---


,crops_vs_weed_weedsgalore_maize_pu,semantic_lizard_glas_set_pu,fgbg_wormbodies_nematodes_pu,fgbg_lidc_malignancy_pu,instance_lizard_glas_set_pu,fgbg_wormbodies_protists_pu,fgbg_lidc_texture_pu,semantic_gta_cityscapes_pu,instance_arctique_nuclei_intensity_pu,semantic_arctique_blood_cells_pu,Mean Rank
Aggregator,,,,,,,,,,,
QFR,0.16,0.27,0.04,0.05,0.26,0.04,0.09,0.06,0.01,0.03,3.30
GMM-F,0.08,0.21,0.10,0.07,0.16,0.06,0.07,0.05,0.05,0.06,5.50
GMM-I,0.09,0.20,0.09,0.07,0.17,0.06,0.06,0.10,0.06,0.07,6.20
GMM-S,0.12,0.20,0.14,0.07,0.18,0.07,0.08,0.05,0.10,0.05,6.50
BCA,0.20,0.33,0.10,0.06,0.30,0.09,0.13,0.04,0.03,0.02,6.90
ATA 0.3,0.29,0.28,0.03,0.12,0.27,0.09,0.14,0.09,0.02,0.03,7.30
PLM 20,0.29,0.33,0.09,0.10,0.28,0.06,0.15,0.09,0.02,0.03,8.10
ATA 0.5,0.29,0.28,0.03,0.19,0.23,0.08,0.15,0.10,0.02,0.03,8.20
PLM 10,0.30,0.31,0.11,0.09,0.27,0.05,0.14,0.09,0.03,0.05,8.80



Successfully saved data to 'eaurc_summary_with_ranks.csv'
